### Test für Batching in ESMPSeudoperplexität:

In [ ]:
import time
import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR
from aguirre.evaluation import ESMPseudoPerplexityScorer

cfg = Config()
sc = ESMPseudoPerplexityScorer(cfg)

seqs = (
    pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"]
    .head(40).tolist()
)
print(f"{len(seqs)} Sequenzen, Laengen {min(map(len, seqs))}-{max(map(len, seqs))}")

t = time.perf_counter()
a = sc._score_schleife(seqs)["pppl_esm2"].to_numpy()
t_schleife = time.perf_counter() - t

t = time.perf_counter()
b = sc._score_batch(seqs)["pppl_esm2"].to_numpy()
t_batch = time.perf_counter() - t

print(f"Schleife {t_schleife:.1f}s | Batch {t_batch:.1f}s | Faktor {t_schleife / t_batch:.1f}")
print("max. abs. Abweichung:", float(np.abs(a - b).max()))
assert np.allclose(a, b, rtol=1e-5)
print("Im Rahmen der Gleitkommagenauigkeit identisch.")

### Test für Batching in ProGen2 Perplexität:

In [ ]:
import time
import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR
from aguirre.evaluation import ProGen2Scorer

cfg = Config()
sc = ProGen2Scorer(cfg)

seqs = (
    pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"]
    .head(40).tolist()
)
print(f"{len(seqs)} Sequenzen, Laengen {min(map(len, seqs))}-{max(map(len, seqs))}")

t = time.perf_counter()
a = sc._score_schleife(seqs)["ppl_progen2"].to_numpy()
t_schleife = time.perf_counter() - t

t = time.perf_counter()
b = sc._score_batch(seqs)["ppl_progen2"].to_numpy()
t_batch = time.perf_counter() - t

print(f"Schleife {t_schleife:.1f}s | Batch {t_batch:.1f}s | Faktor {t_schleife / t_batch:.1f}")
print("max. rel. Abweichung:", float((np.abs(a - b) / a).max()))
assert np.allclose(a, b, rtol=1e-4)

40 Sequenzen, Laengen 7-49


ProGen2 (Schleife):   0%|          | 0/40 [00:00<?, ?it/s]

ProGen2:   0%|          | 0/2 [00:00<?, ?it/s]

Schleife 8.1s | Batch 2.6s | Faktor 3.1
max. rel. Abweichung: 4.100883644480403e-05


In [2]:
seqs = pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"].head(200).tolist()
for bs in (8, 16, 32, 64, 128):
    sc.batch_size = bs
    t = time.perf_counter(); sc._score_batch(seqs); dt = time.perf_counter() - t
    print(f"batch_size {bs:4d}: {dt:5.1f}s  ->  {dt/len(seqs)*1000:5.1f} ms/Sequenz")

ProGen2:   0%|          | 0/25 [00:00<?, ?it/s]

batch_size    8:   9.2s  ->   46.0 ms/Sequenz


ProGen2:   0%|          | 0/13 [00:00<?, ?it/s]

batch_size   16:   6.0s  ->   29.9 ms/Sequenz


ProGen2:   0%|          | 0/7 [00:00<?, ?it/s]

batch_size   32:   5.6s  ->   28.0 ms/Sequenz


ProGen2:   0%|          | 0/4 [00:00<?, ?it/s]

batch_size   64:   6.1s  ->   30.4 ms/Sequenz


ProGen2:   0%|          | 0/2 [00:00<?, ?it/s]

batch_size  128:   7.2s  ->   36.2 ms/Sequenz


In [3]:
kurz = seqs[int(np.argmin([len(s) for s in seqs]))]
lang = seqs[int(np.argmax([len(s) for s in seqs]))]

allein = sc._score_batch([kurz])["ppl_progen2"].iloc[0]
zusammen = sc._score_batch([kurz, lang])["ppl_progen2"].iloc[0]
print(f"allein {allein:.6f} | neben {len(lang)}er {zusammen:.6f} | "
      f"rel. Diff {abs(allein-zusammen)/allein:.2e}")
assert np.isclose(allein, zusammen, rtol=1e-4)

ProGen2:   0%|          | 0/1 [00:00<?, ?it/s]

ProGen2:   0%|          | 0/1 [00:00<?, ?it/s]

allein 91.591220 | neben 50er 91.590565 | rel. Diff 7.15e-06


### Vorbereitung für KnownAMPSimilirityScorer

In [ ]:
from aguirre.data import write_fasta
import pandas as pd
write_fasta(pd.read_csv(PATH)["sequence"].tolist(), TRACKED_DIR / "torres_train.fasta")